# Automated Colab evaluation

Chọn **Runtime → Run all** để chạy toàn bộ pipeline. Các cell được tách theo từng giai đoạn để dễ theo dõi, chạy lại và tìm lỗi.

> Google Colab vẫn yêu cầu xác nhận quyền truy cập Drive/Sheets; đây là bước bảo mật không thể tự động bỏ qua.


In [1]:
#@title 1. Cấu hình chạy
REPO_URL = "https://github.com/doantrunghieu08/optimization_model_monocular_3.git"  #@param {type:"string"}
REPO_BRANCH = "version_raycasting"  #@param {type:"string"}
DATA_ROOT = "/content/drive/MyDrive/belief_driven_occlusion_detection_and_error_correction_4_3DHPE_using_uncalibrated_dual_cams"  #@param {type:"string"}
SUBJECTS = "S8"  #@param {type:"string"}
SEQUENCES = "Seq1"  #@param {type:"string"}
SEGMENTS = "*"  #@param {type:"string"}
EXCLUDED_CAMERAS = "1"  #@param {type:"string"}
ENABLE_LEARNABLE = False  #@param {type:"boolean"}
ENABLE_LEARNABLE_EXTRA = True  #@param {type:"boolean"}
SPREADSHEET_NAME = ""  #@param {type:"string"}
NOTEBOOK_NAME = "ablation_hieuDT_belief_fusion_H260912RayCasting_optical_global_kinematic_mse_alpha1E_1_beta85E_2.ipynb"  #@param {type:"string"}
SMPL_NEUTRAL_FILE_ID = "1xblXsbK1rTSn5cG934cDhRFB0Apn64Ll"  #@param {type:"string"}
SMPL_PART_SEGMENTATION_FILE_ID = "19w6RSoqdCJwUMu8wd1tYiF7uqLMO19BD"  #@param {type:"string"}

In [2]:
#@title 2. Chuẩn bị repository và môi trường
import importlib
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path



def run(command, cwd=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)


repo_dir = Path("/content/optimization_model_monocular_3")
if (repo_dir / ".git").exists():
    if (repo_dir / ".git").exists():
      print(f"Updating branch: {REPO_BRANCH}")

    # Fetch branch và tạo/cập nhật chính xác remote-tracking ref
    run([
        "git",
        "fetch",
        "origin",
        f"{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}"
    ], repo_dir)

    # Tạo/reset local branch theo remote
    run([
        "git",
        "checkout",
        "-f",
        "-B",
        REPO_BRANCH,
        f"origin/{REPO_BRANCH}"
    ], repo_dir)
elif repo_dir.exists():
    backup_dir = repo_dir.with_name(f"{repo_dir.name}_backup_{os.getpid()}")
    repo_dir.rename(backup_dir)
    print(f"Moved the existing non-Git directory to {backup_dir}")
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])

# Install everything before importing project/Colab dependencies.
requirements_path = repo_dir / "requirements.txt"
if sys.version_info >= (3, 13):
    requirements_lines = requirements_path.read_text(encoding="utf-8").splitlines()
    requirements_lines = [
        line for line in requirements_lines
        if not re.match(r"^\s*(numpy|scipy)(?:[<>=!~;]|$)", line, re.IGNORECASE)
    ]
    requirements_lines.extend(["numpy>=2.1,<2.3", "scipy>=1.14.1,<1.15"])
    requirements_path = Path("/tmp/requirements-colab.txt")
    requirements_path.write_text("\n".join(requirements_lines) + "\n", encoding="utf-8")

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
run([
    sys.executable, "-m", "pip", "install", "-q",
    "gspread", "google-api-python-client", "pandas", "gdown", "PyYAML",
])

scientific_check = [
    sys.executable,
    "-c",
    "import numpy, scipy; from scipy.spatial.distance import cdist; "
    "print('NumPy', numpy.__version__, '| SciPy', scipy.__version__)",
]
check_result = subprocess.run(scientific_check, text=True, capture_output=True)
if check_result.returncode:
    print("Repairing the incompatible NumPy/SciPy installation...")
    scientific_specs = (
        ["numpy>=2.1,<2.3", "scipy>=1.14.1,<1.15"]
        if sys.version_info >= (3, 13)
        else ["numpy<2", "scipy<2"]
    )
    run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "--force-reinstall", "--no-cache-dir", *scientific_specs,
    ])
    run(scientific_check)
else:
    print(check_result.stdout.strip())

if shutil.which("ffmpeg") is None:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "ffmpeg"])
importlib.invalidate_caches()


$ git clone --depth 1 --branch version_raycasting https://github.com/doantrunghieu08/optimization_model_monocular_3.git /content/optimization_model_monocular_3
$ /usr/bin/python3 -m pip install -q -r /tmp/requirements-colab.txt
$ /usr/bin/python3 -m pip install -q gspread google-api-python-client pandas gdown PyYAML
NumPy 2.1.3 | SciPy 1.14.1


In [3]:
#@title 3. Kết nối Google Drive và tải model
from google.colab import auth, drive
from google.auth import default
from googleapiclient.discovery import build
import gdown
import yaml
from ruamel.yaml import YAML

drive.mount("/content/drive", force_remount=False)
auth.authenticate_user()
credentials, _ = default()

try:
    user = build("drive", "v3", credentials=credentials).about().get(fields="user").execute()["user"]
    os.environ["RUNNER_NAME"] = user.get("displayName", "Colab_User")
    os.environ["RUNNER_EMAIL"] = user.get("emailAddress", "")
except Exception as exc:
    print(f"Could not read Drive profile ({exc}); using Colab_User.")
    os.environ["RUNNER_NAME"] = "Colab_User"

model_path = repo_dir / "models" / "SMPL_NEUTRAL.pkl"
model_path.parent.mkdir(parents=True, exist_ok=True)
if not model_path.exists() or model_path.stat().st_size < 1_000_000:
    print("Downloading SMPL_NEUTRAL.pkl...")
    downloaded = gdown.download(id=SMPL_NEUTRAL_FILE_ID, output=str(model_path), quiet=False)
    if not downloaded or not model_path.exists():
        raise RuntimeError("Could not download SMPL_NEUTRAL.pkl. Check the Drive file ID/access permission.")

segmentation_path = repo_dir / "models" / "smpl_partSegmentation_mapping.pkl"
if not segmentation_path.exists() or segmentation_path.stat().st_size == 0:
    print("Downloading smpl_partSegmentation_mapping.pkl...")
    downloaded = gdown.download(id=SMPL_PART_SEGMENTATION_FILE_ID, output=str(segmentation_path), quiet=False)
    if not downloaded or not segmentation_path.exists() or segmentation_path.stat().st_size == 0:
        raise RuntimeError("Could not download smpl_partSegmentation_mapping.pkl. Check the Drive file ID/access permission.")


Downloading...
From: https://drive.google.com/uc?id=1xblXsbK1rTSn5cG934cDhRFB0Apn64Ll
To: /content/optimization_model_monocular_3/models/SMPL_NEUTRAL.pkl
100%|██████████| 39.0M/39.0M [00:01<00:00, 34.8MB/s]


Downloading...
From: https://drive.google.com/uc?id=19w6RSoqdCJwUMu8wd1tYiF7uqLMO19BD
To: /content/optimization_model_monocular_3/models/smpl_partSegmentation_mapping.pkl
100%|██████████| 27.7k/27.7k [00:00<00:00, 12.6MB/s]


In [4]:
#@title 4. Áp dụng cấu hình pipeline
# Các tham số fusion (ALPHA, BETA, LOCAL_METHOD, GLOBAL, KINEMATIC_CONSTRAINTS, LOSS_TYPE)
# được tự động đọc từ tên notebook bởi set_env_from_filename() trong run_brute_force().
# Chỉ cần set các flag learnable ở đây vì chúng không được encode trong tên file.

import os
os.environ["NOTEBOOK_NAME"]          = NOTEBOOK_NAME
os.environ["ENABLE_LEARNABLE"]       = str(ENABLE_LEARNABLE).lower()
os.environ["ENABLE_LEARNABLE_EXTRA"] = str(ENABLE_LEARNABLE_EXTRA).lower()

# Patch pipeline.yml: bật optimizer cho ablation và cập nhật learnable flags.
from pathlib import Path
from ruamel.yaml import YAML

pipeline_path = repo_dir / "configs" / "pipeline.yml"
roundtrip_yaml = YAML()
with pipeline_path.open("r", encoding="utf-8") as stream:
    pipeline_config = roundtrip_yaml.load(stream)
pipeline_config["learnable"]["enabled"]       = bool(ENABLE_LEARNABLE)
pipeline_config["learnable_extra"]["enabled"] = bool(ENABLE_LEARNABLE_EXTRA)
pipeline_config["fusion"]["optimization"]["enabled"] = True
with pipeline_path.open("w", encoding="utf-8") as stream:
    roundtrip_yaml.dump(pipeline_config, stream)

keypoint_map_path = repo_dir / "configs" / "keypoints3D_map.yml"
with keypoint_map_path.open("r", encoding="utf-8") as stream:
    keypoint_map = roundtrip_yaml.load(stream)
all_joint_names = [item["name"] for item in keypoint_map["keypoints"]]
if len(keypoint_map.get("priority2", [])) != len(all_joint_names):
    keypoint_map["priority2"] = all_joint_names
    with keypoint_map_path.open("w", encoding="utf-8") as stream:
        roundtrip_yaml.dump(keypoint_map, stream)

In [5]:
#@title 5. Dò dữ liệu và tạo brute_force.yml
def accepts(filter_text, value):
    requested = {item.strip() for item in filter_text.split(",") if item.strip()}
    return not requested or "*" in requested or value in requested


def find_video(image_sequence, segments_dir, pkl_path, camera_id, segment_id):
    candidates = [
        pkl_path.parent / "output.mp4",
        pkl_path.parent / f"video_{camera_id}_seg_{segment_id}.mp4",
        segments_dir / f"video_{camera_id}_seg_{segment_id}.mp4",
        image_sequence / f"video_{camera_id}.avi",
        image_sequence / f"video_{camera_id}.mp4",
    ]
    return next((path for path in candidates if path.exists()), None)


data_root = Path(DATA_ROOT).expanduser()
if not data_root.exists():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {data_root}")

image_sequences = [data_root] if data_root.name == "imageSequence" else sorted(data_root.rglob("imageSequence"))
excluded_cameras = {item.strip() for item in EXCLUDED_CAMERAS.split(",") if item.strip()}
discovered_segments = []

for image_sequence in image_sequences:
    sequence = image_sequence.parent.name
    subject = image_sequence.parent.parent.name
    if not accepts(SUBJECTS, subject) or not accepts(SEQUENCES, sequence):
        continue

    segments_dir = next((image_sequence / name for name in ("Segments", "segments") if (image_sequence / name).is_dir()), None)
    gt_dir = next((path for path in (image_sequence / "GT", image_sequence.parent / "GT") if path.is_dir()), None)
    if segments_dir is None or gt_dir is None:
        print(f"Skipping {subject}/{sequence}: missing Segments or GT directory.")
        continue

    grouped = {}
    for pkl_path in sorted(segments_dir.rglob("*.pkl")):
        match = re.search(r"video_(\d+)_seg_(\d+)$", pkl_path.stem)
        if not match:
            continue
        camera_id, segment_id = match.groups()
        segment_name = f"seg_{segment_id}"
        if camera_id in excluded_cameras or not accepts(SEGMENTS, segment_name):
            continue
        if not (gt_dir / f"video_{camera_id}_{segment_name}.json").exists():
            continue
        video_path = find_video(image_sequence, segments_dir, pkl_path, camera_id, segment_id)
        if video_path is None:
            continue
        grouped.setdefault(segment_name, {})[camera_id] = {
            "id": f"video_{camera_id}_{segment_name}",
            "pkl": str(pkl_path.resolve()),
            "video": str(video_path.resolve()),
        }

    for segment_name, cameras_by_id in sorted(grouped.items()):
        cameras = [cameras_by_id[key] for key in sorted(cameras_by_id, key=int)]
        if len(cameras) >= 2:
            discovered_segments.append({
                "name": f"{subject}_{sequence}_{segment_name}",
                "ground_truth_dir": str(gt_dir.resolve()),
                "cameras": cameras,
            })

if not discovered_segments:
    raise RuntimeError(
        "No runnable segment with at least two cameras was found. "
        "Expected PKL names like video_0_seg_1.pkl and matching GT JSON files."
    )

brute_force_path = repo_dir / "configs" / "brute_force.yml"
with brute_force_path.open("w", encoding="utf-8") as stream:
    yaml.safe_dump({"segments": discovered_segments}, stream, sort_keys=False, allow_unicode=True)

camera_count = sum(len(segment["cameras"]) for segment in discovered_segments)
pair_count = sum(len(segment["cameras"]) * (len(segment["cameras"]) - 1) for segment in discovered_segments)
print(f"Discovered {len(discovered_segments)} segments, {camera_count} camera inputs, {pair_count} ordered pairs.")
print(f"Fusion config: notebook filename; learnable={ENABLE_LEARNABLE}, learnable_extra={ENABLE_LEARNABLE_EXTRA}")


Discovered 7 segments, 42 camera inputs, 210 ordered pairs.
Fusion config: notebook filename; learnable=False, learnable_extra=True


In [ ]:
#@title 6. Chạy đánh giá
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
import brute_force_runner

# Avoid the runner's timed input: blank means its deterministic per-user default.
brute_force_runner.get_spreadsheet_name_input = (
    lambda default_name, timeout=10: SPREADSHEET_NAME.strip() or Path(NOTEBOOK_NAME).stem
)
brute_force_runner.run_brute_force()
print("✅ Finished. The final Google Sheets URL is printed above.")


/content/optimization_model_monocular_3/compat.py:10: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'object'): np.object = object
/content/optimization_model_monocular_3/compat.py:14: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'str'): np.str = str


[ENV] Detecting notebook: 'ablation_hieuDT_belief_fusion_H260912RayCasting_optical_global_kinematic_mse_alpha1E_1_beta85E_2.ipynb'
[ENV] ALPHA=0.1  (raw: '1E_1')
[ENV] BETA=0.85   (raw: '85E_2')
[ENV] LOCAL_METHOD=optical_aware_belief
[ENV] GLOBAL=true
[ENV] KINEMATIC_CONSTRAINTS=true
[ENV] LOSS_TYPE=mse
Alpha trong config: 0.1
Beta trong config: 0.85
Global belief trong config: True
Local method trong config: optical_aware_belief
Kinematic constraints trong config: True
Loss type trong config: mse
Optimization enabled trong config: True
Orientation correction trong config: False
Learnable trong config: False
Learnable extra trong config: True
[!] Config hiện tại khác worksheet chưa hoàn thành. Tạo worksheet mới để không trộn kết quả cũ.
[+] Khởi tạo worksheet (cell/tab) mới: 'Run_2026-09-17_09-33-01' trong file Google Sheets 'ablation_hieuDT_belief_fusion_H260912RayCasting_optical_global_kinematic_mse_alpha1E_1_beta85E_2'.

=== Bắt đầu vét cạn cho Segment: S8_Seq1_seg_1 (30 cặp) ===



[Pose] Exporting JSONs: 100%|██████████| 424/424 [00:09<00:00, 42.83frame/s]


[Pose] Done. Output: /content/optimization_model_monocular_3/output/pose_results
[Pipeline] Running fusion with offset=0
[Fusion] Camera-space mesh cache: 424 synced frames
[Fusion] Torso ray-casting mesh: 3148 triangles
[Fusion] Found 424 pose JSON files


[Fusion] Processing:   0%|          | 0/424 [00:00<?, ?frame/s]

...

[LearnableExtra] Saving JSON Results: 100%|██████████| 335/335 [00:00<00:00, 752.66it/s]


[LearnableExtra] Done. Output: /content/optimization_model_monocular_3/output/learnable_extra_results
[Pipeline] Running evaluation with offset=1
[Evaluation] Đã lưu metrics vào os.environ['LEARNABLE_EXTRA_METRICS']: {"camera1": {"MPJPE": "120.50", "PA-MPJPE": "61.38"}, "camera2": {"MPJPE": "127.04", "PA-MPJPE": "70.72"}}
[Evaluation] Done. Output: /content/optimization_model_monocular_3/output/evaluation_results
[2026-09-17 11:26:49] Kết quả video_7_seg_7-video_4_seg_7: MPJPE=99.45 (Δ -0.03%), PA-MPJPE=54.11 (Δ -0.04%), MBLE=33.93, Accel=18.38 LE MPJPE=120.50 (LE PA-MPJPE 61.38), 

--- [Tiến trình: 174/210] Master=video_7_seg_7 | Supplement=video_5_seg_7 ---
[Preprocess] Start | output=/content/optimization_model_monocular_3/output/preprocess_results
[Preprocess] Offset input | cam1=video_7_seg_7.pkl | cam2=video_5_seg_7.pkl
[Preprocess] Offset=0
[Preprocess] Exported 2D keypoints to data_cam1.json
[Preprocess] Exported 2D keypoints to data_cam2.json
[Preprocess] Selected offset | off

[Pose] Exporting JSONs: 100%|██████████| 492/492 [00:09<00:00, 53.09frame/s]


[Pose] Done. Output: /content/optimization_model_monocular_3/output/pose_results
[Pipeline] Running fusion with offset=0
[Fusion] Camera-space mesh cache: 492 synced frames
[Fusion] Torso ray-casting mesh: 3148 triangles
[Fusion] Found 492 pose JSON files


[Fusion] Processing:   0%|          | 0/492 [00:00<?, ?frame/s]